# Sudoku por Programação Inteira com Python-MIP

Este notebook recebe um problema de Sudoku 9×9 qualquer e o resolve formulando-o como um
problema de **Programação Linear Inteira**, resolvido pelo [Python-MIP](https://www.python-mip.com/)
com o solver CBC.

## Modelo

### Variáveis de decisão — 729 binárias

$$
x_{K,i,j} = \begin{cases} 1, & \text{se o número } K \text{ aparece na célula } (i,j)\\ 0, & \text{caso contrário}\end{cases}
\qquad K, i, j \in \{1,\dots,9\}
$$

No código cada variável recebe o nome `x_K_i_j` (ex.: `x_5_1_1` = "o número 5 está na linha 1, coluna 1").

### Pistas — variáveis fixadas em 1

Para cada célula $(i,j)$ já preenchida com o número $K$ no enunciado, fixamos $x_{K,i,j} = 1$
por meio dos limites da variável (`lb = ub = 1`). As demais variáveis daquela célula ficam
iguais a 0 por consequência da restrição (1).

### Restrições — 4 grupos × 81 = 324

| # | Grupo | Restrição | Para todo | Qtde |
|---|-------|-----------|-----------|------|
| 1 | Em cada célula aparece exatamente 1 número | $\sum_{K=1}^{9} x_{K,i,j} = 1$ | $i, j$ | 81 |
| 2 | Em cada linha o número $K$ aparece exatamente 1 vez | $\sum_{j=1}^{9} x_{K,i,j} = 1$ | $K, i$ | 81 |
| 3 | Em cada coluna o número $K$ aparece exatamente 1 vez | $\sum_{i=1}^{9} x_{K,i,j} = 1$ | $K, j$ | 81 |
| 4 | Em cada subquadrado o número $K$ aparece exatamente 1 vez | $\sum_{(i,j)\in Q} x_{K,i,j} = 1$ | $K$, subquadrado $Q$ | 81 |

### Função objetivo

Nenhuma — é um **problema de viabilidade**: qualquer solução inteira que satisfaça as 324
restrições é uma solução do Sudoku. Se as pistas forem incompatíveis, o solver devolve
`INFEASIBLE`.

## 1. Instalação e importações

In [1]:
try:
    import mip
except ImportError:          # no Google Colab o pacote não vem instalado
    %pip install -q mip
    import mip

from mip import BINARY, Model, OptimizationStatus, SearchEmphasis, xsum
print("Python-MIP", mip.__version__)

Python-MIP 2.0.0


## 2. Leitura e exibição da grade

In [2]:
N = 9                       # dimensão da grade
B = 3                       # dimensão do subquadrado
VALORES = range(1, N + 1)   # K: números 1..9
LINHAS  = range(1, N + 1)   # i: linhas 1..9
COLUNAS = range(1, N + 1)   # j: colunas 1..9

# Os 9 subquadrados 3×3 (numerados de 1 a 9, da esquerda para a direita, de cima para baixo):
# cada um é o par (linhas, colunas) que o compõe.
SUBQUADRADOS = [(range(bi*B + 1, bi*B + B + 1), range(bj*B + 1, bj*B + B + 1))
                for bi in range(B) for bj in range(B)]


def ler_grade(texto):
    '''Converte um texto com 81 símbolos em uma grade 9×9 (lista de listas; 0 = vazio).
    Dígitos 1-9 são pistas; 0 . _ * ? marcam célula vazia; espaços e | - + = são ignorados.'''
    simbolos = []
    for ch in texto:
        if ch.isdigit():
            simbolos.append(int(ch))
        elif ch in "._*?":
            simbolos.append(0)
    if len(simbolos) != N * N:
        raise ValueError(f"Esperados {N*N} símbolos, encontrados {len(simbolos)}.")
    return [simbolos[r*N:(r+1)*N] for r in range(N)]


def formatar_grade(grade, vazio="."):
    '''Grade como texto, com os subquadrados separados.'''
    linhas = []
    for r in range(N):
        if r and r % B == 0:
            linhas.append("------+-------+------")
        cel = [str(v) if v else vazio for v in grade[r]]
        linhas.append(" | ".join(" ".join(cel[c:c+B]) for c in range(0, N, B)))
    return "\n".join(linhas)


def lado_a_lado(esq, dir_, titulo_esq="Problema", titulo_dir="Solução", sep="      "):
    le, ld = esq.splitlines(), dir_.splitlines()
    larg = max(len(l) for l in le)
    return "\n".join([f"{titulo_esq:<{larg}}{sep}{titulo_dir}"] +
                     [f"{a:<{larg}}{sep}{b}" for a, b in zip(le, ld)])

## 3. O modelo de Programação Inteira

A função abaixo monta o modelo: cria as **729 variáveis binárias** `x_K_i_j`, **fixa em 1** as
que correspondem às pistas e adiciona os **4 grupos de 81 restrições**.

In [3]:
def construir_modelo(grade):
    '''Monta o modelo PLI do Sudoku. Devolve (modelo, x), com x[(K, i, j)] = variável x_K_i_j.'''
    m = Model("sudoku")                       # sem função objetivo: problema de viabilidade
    m.verbose = 0
    m.emphasis = SearchEmphasis.FEASIBILITY   # basta encontrar uma solução viável

    # ---- 729 variáveis binárias x_K_i_j ---------------------------------------------
    x = {(k, i, j): m.add_var(name=f"x_{k}_{i}_{j}", var_type=BINARY)
         for k in VALORES for i in LINHAS for j in COLUNAS}

    # ---- Pistas: variáveis fixadas em 1 ---------------------------------------------
    for i in LINHAS:
        for j in COLUNAS:
            k = grade[i-1][j-1]
            if k:
                x[k, i, j].lb = 1             # lb = ub = 1  =>  x_K_i_j = 1

    # ---- (1) Em cada célula aparece exatamente 1 número ----------------- 81 restrições
    for i in LINHAS:
        for j in COLUNAS:
            m += xsum(x[k, i, j] for k in VALORES) == 1, f"celula_{i}_{j}"

    # ---- (2) Em cada linha o número K aparece exatamente 1 vez ---------- 81 restrições
    for k in VALORES:
        for i in LINHAS:
            m += xsum(x[k, i, j] for j in COLUNAS) == 1, f"linha_{k}_{i}"

    # ---- (3) Em cada coluna o número K aparece exatamente 1 vez --------- 81 restrições
    for k in VALORES:
        for j in COLUNAS:
            m += xsum(x[k, i, j] for i in LINHAS) == 1, f"coluna_{k}_{j}"

    # ---- (4) Em cada subquadrado o número K aparece exatamente 1 vez ---- 81 restrições
    for k in VALORES:
        for q, (li, lj) in enumerate(SUBQUADRADOS, start=1):
            m += xsum(x[k, i, j] for i in li for j in lj) == 1, f"subquadrado_{k}_{q}"

    return m, x


def extrair_solucao(x):
    '''Lê os valores das variáveis e devolve a grade resolvida.'''
    sol = [[0]*N for _ in range(N)]
    for (k, i, j), var in x.items():
        if var.x is not None and var.x > 0.5:
            sol[i-1][j-1] = k
    return sol


def verificar(solucao, grade=None):
    '''Confere (sem usar o solver) se a solução é um Sudoku válido e respeita as pistas.'''
    alvo = set(VALORES)
    for r in range(N):
        if set(solucao[r]) != alvo or {solucao[c][r] for c in range(N)} != alvo:
            return False
    for li, lj in SUBQUADRADOS:
        if {solucao[i-1][j-1] for i in li for j in lj} != alvo:
            return False
    if grade is not None:
        if any(grade[r][c] and grade[r][c] != solucao[r][c] for r in range(N) for c in range(N)):
            return False
    return True


def resolver_sudoku(texto, max_segundos=60):
    '''Recebe o problema como texto, resolve e mostra o resultado. Devolve a grade solução (ou None).'''
    grade = ler_grade(texto)
    m, x = construir_modelo(grade)
    status = m.optimize(max_seconds=max_segundos)

    n_pistas = sum(1 for linha in grade for v in linha if v)
    print(f"Modelo : {m.num_cols} variáveis binárias, {m.num_rows} restrições, "
          f"{n_pistas} variáveis fixadas em 1 (pistas)")
    print(f"Status : {status.name}\n")

    if status not in (OptimizationStatus.OPTIMAL, OptimizationStatus.FEASIBLE):
        print(formatar_grade(grade))
        print("\nO problema é INVIÁVEL: as pistas são incompatíveis entre si.")
        return None

    solucao = extrair_solucao(x)
    print(lado_a_lado(formatar_grade(grade), formatar_grade(solucao)))
    print("\nVerificação independente:", "OK" if verificar(solucao, grade) else "FALHOU")
    return solucao

## 4. Resolvendo um Sudoku

O problema pode ser informado em qualquer formato com 81 símbolos: 9 linhas de 9, uma única
linha de 81 caracteres, com ou sem separadores.

In [4]:
classico = '''
5 3 . | . 7 . | . . .
6 . . | 1 9 5 | . . .
. 9 8 | . . . | . 6 .
------+-------+------
8 . . | . 6 . | . . 3
4 . . | 8 . 3 | . . 1
7 . . | . 2 . | . . 6
------+-------+------
. 6 . | . . . | 2 8 .
. . . | 4 1 9 | . . 5
. . . | . 8 . | . 7 9
'''
solucao = resolver_sudoku(classico)

Modelo : 729 variáveis binárias, 324 restrições, 30 variáveis fixadas em 1 (pistas)
Status : OPTIMAL

Problema                   Solução
5 3 . | . 7 . | . . .      5 3 4 | 6 7 8 | 9 1 2
6 . . | 1 9 5 | . . .      6 7 2 | 1 9 5 | 3 4 8
. 9 8 | . . . | . 6 .      1 9 8 | 3 4 2 | 5 6 7
------+-------+------      ------+-------+------
8 . . | . 6 . | . . 3      8 5 9 | 7 6 1 | 4 2 3
4 . . | 8 . 3 | . . 1      4 2 6 | 8 5 3 | 7 9 1
7 . . | . 2 . | . . 6      7 1 3 | 9 2 4 | 8 5 6
------+-------+------      ------+-------+------
. 6 . | . . . | 2 8 .      9 6 1 | 5 3 7 | 2 8 4
. . . | 4 1 9 | . . 5      2 8 7 | 4 1 9 | 6 3 5
. . . | . 8 . | . 7 9      3 4 5 | 2 8 6 | 1 7 9

Verificação independente: OK


### Um Sudoku difícil

O Sudoku abaixo foi publicado em 2012 pelo matemático finlandês Arto Inkala como "o mais
difícil do mundo" (21 pistas). Para o modelo de Programação Inteira a dificuldade é
irrelevante: a resolução leva o mesmo tempo.

In [5]:
dificil = '''
8........
..36.....
.7..9.2..
.5...7...
....457..
...1...3.
..1....68
..85...1.
.9....4..
'''
solucao = resolver_sudoku(dificil)

Modelo : 729 variáveis binárias, 324 restrições, 21 variáveis fixadas em 1 (pistas)
Status : OPTIMAL

Problema                   Solução
8 . . | . . . | . . .      8 1 2 | 7 5 3 | 6 4 9
. . 3 | 6 . . | . . .      9 4 3 | 6 8 2 | 1 7 5
. 7 . | . 9 . | 2 . .      6 7 5 | 4 9 1 | 2 8 3
------+-------+------      ------+-------+------
. 5 . | . . 7 | . . .      1 5 4 | 2 3 7 | 8 9 6
. . . | . 4 5 | 7 . .      3 6 9 | 8 4 5 | 7 2 1
. . . | 1 . . | . 3 .      2 8 7 | 1 6 9 | 5 3 4
------+-------+------      ------+-------+------
. . 1 | . . . | . 6 8      5 2 1 | 9 7 4 | 3 6 8
. . 8 | 5 . . | . 1 .      4 3 8 | 5 2 6 | 9 1 7
. 9 . | . . . | 4 . .      7 9 6 | 3 1 8 | 4 5 2

Verificação independente: OK


### Um Sudoku sem solução

Se as pistas forem incompatíveis (abaixo, dois 5 na primeira linha), não existe atribuição
que satisfaça as 324 restrições e o solver devolve o status `INFEASIBLE`.

In [6]:
# o exemplo clássico com um 5 a mais no fim da primeira linha
inviavel = classico.replace("5 3 . | . 7 . | . . .", "5 3 . | . 7 . | . . 5", 1)
solucao = resolver_sudoku(inviavel)

Modelo : 729 variáveis binárias, 324 restrições, 31 variáveis fixadas em 1 (pistas)
Status : INFEASIBLE

5 3 . | . 7 . | . . 5
6 . . | 1 9 5 | . . .
. 9 8 | . . . | . 6 .
------+-------+------
8 . . | . 6 . | . . 3
4 . . | 8 . 3 | . . 1
7 . . | . 2 . | . . 6
------+-------+------
. 6 . | . . . | 2 8 .
. . . | 4 1 9 | . . 5
. . . | . 8 . | . 7 9

O problema é INVIÁVEL: as pistas são incompatíveis entre si.


## 5. Inspecionando o modelo

Exportar o modelo em formato LP permite ver as 729 variáveis, as 324 restrições (81 de cada
grupo) e as pistas fixadas na seção `Bounds` como `1 <= x_K_i_j <= 1`.

In [7]:
m, x = construir_modelo(ler_grade(classico))
m.write("sudoku_classico.lp")

with open("sudoku_classico.lp") as f:
    linhas = f.read().splitlines()

print(f"{m.num_cols} variáveis, {m.num_rows} restrições")
for grupo in ("celula_", "linha_", "coluna_", "subquadrado_"):
    print(f"  {grupo:<13}{sum(l.startswith(grupo) for l in linhas)} restrições")
print(f"  pistas fixadas: {sum(' 1 <= x_' in l for l in linhas)}")

print("\nPrimeiras linhas do arquivo LP:")
print("\n".join(linhas[:7]))
print("...")
print("\n".join(l for l in linhas if l.startswith(("linha_5_1:", "coluna_5_1:", "subquadrado_5_1:"))))
print("...")
fixadas = [l for l in linhas if " 1 <= x_" in l]
print("\n".join(fixadas[:5]))
print(f"... ({len(fixadas)} variáveis fixadas em 1 ao todo)")

729 variáveis, 324 restrições
  celula_      81 restrições
  linha_       81 restrições
  coluna_      81 restrições
  subquadrado_ 81 restrições
  pistas fixadas: 30

Primeiras linhas do arquivo LP:
\Problem name: sudoku

Minimize
OBJROW:
Subject To
celula_1_1:  x_1_1_1 + x_2_1_1 + x_3_1_1 + x_4_1_1 + x_5_1_1 + x_6_1_1 + x_7_1_1 + x_8_1_1 + x_9_1_1 = 1
celula_1_2:  x_1_1_2 + x_2_1_2 + x_3_1_2 + x_4_1_2 + x_5_1_2 + x_6_1_2 + x_7_1_2 + x_8_1_2 + x_9_1_2 = 1
...
linha_5_1:  x_5_1_1 + x_5_1_2 + x_5_1_3 + x_5_1_4 + x_5_1_5 + x_5_1_6 + x_5_1_7 + x_5_1_8 + x_5_1_9 = 1
coluna_5_1:  x_5_1_1 + x_5_2_1 + x_5_3_1 + x_5_4_1 + x_5_5_1 + x_5_6_1 + x_5_7_1 + x_5_8_1 + x_5_9_1 = 1
subquadrado_5_1:  x_5_1_1 + x_5_1_2 + x_5_1_3 + x_5_2_1 + x_5_2_2 + x_5_2_3 + x_5_3_1 + x_5_3_2 + x_5_3_3 = 1
...
 1 <= x_1_2_4 <= 1
 1 <= x_1_5_9 <= 1
 1 <= x_1_8_5 <= 1
 1 <= x_2_6_5 <= 1
 1 <= x_2_7_7 <= 1
... (30 variáveis fixadas em 1 ao todo)


## 6. A solução é única?

Um bom Sudoku tem solução única. Para verificar, depois de encontrar uma solução acrescentamos
ao modelo um **corte** que a exclui — a soma das 81 variáveis que valem 1 nessa solução deve
ser $\le 80$ — e resolvemos de novo. Se o novo modelo for inviável, a solução é única.
(Esse corte é adicional ao modelo básico de 324 restrições.)

In [8]:
def enumerar_solucoes(texto, maximo=2):
    '''Devolve até `maximo` soluções distintas do Sudoku.'''
    m, x = construir_modelo(ler_grade(texto))
    solucoes = []
    while len(solucoes) < maximo:
        if m.optimize() not in (OptimizationStatus.OPTIMAL, OptimizationStatus.FEASIBLE):
            break
        sol = extrair_solucao(x)
        solucoes.append(sol)
        m += xsum(x[sol[i-1][j-1], i, j] for i in LINHAS for j in COLUNAS) <= N*N - 1   # corte
    return solucoes


for nome, texto in [("Clássico", classico), ("Difícil", dificil), ("Grade vazia", "." * 81)]:
    n = len(enumerar_solucoes(texto, maximo=2))
    print(f"{nome:<12}: {'solução ÚNICA' if n == 1 else 'MAIS DE UMA solução' if n > 1 else 'sem solução'}")

Clássico    : solução ÚNICA


Difícil     : solução ÚNICA


Grade vazia : MAIS DE UMA solução


## Conclusão

O Sudoku foi modelado como um problema de viabilidade em Programação Linear Inteira com
**729 variáveis binárias** $x_{K,i,j}$ e **4 × 81 = 324 restrições** de igualdade; as pistas
entram no modelo como variáveis fixadas em 1. O solver CBC, via Python-MIP, encontra a solução
de qualquer Sudoku válido em fração de segundo e, quando as pistas são incompatíveis, prova
que o problema é inviável.